In [1]:
from datasets import load_dataset
import sentencepiece as spm

from opencc import OpenCC

cc = OpenCC('t2s')

import os
import re

from tqdm import tqdm

from dotenv import load_dotenv

load_dotenv()

d:\programs\endfield\translator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Download & Load Dataset

In [2]:
ds = load_dataset(
    "wikimedia/wikipedia",
    "20231101.zh",
    split="train"
)

## Set Variables

In [3]:
CHARACTER_MAP = "gkamztlbdqiyfucxbhsjoprnweygtjmevchdxsanqolkrvwiypjzquhe"

def encode_line(text: str) -> str:
    return "".join(CHARACTER_MAP[ord(c) % 56] if c not in "\n\t\r " else c for c in text)

MAX_CHARS = float('inf')
src_path = "corpus/train.zh"
tgt_path = "corpus/train.skz"
os.makedirs("corpus", exist_ok=True)

## Generate Corpus

In [4]:
total_chars = 0

with open(src_path, "w", encoding="utf-8") as f_src:
    for item in tqdm(ds, desc="处理语料", unit="item"):
        txt = re.sub(r"\s+", " ", item["text"]).strip()
        if len(txt) < 20 or len(txt) > 200:
            continue

        f_src.write(txt + "\n")
        
        total_chars += len(txt)
        if total_chars >= MAX_CHARS:
            break

with open(src_path, 'r', encoding='utf-8') as f_src:
    src_string = f_src.read()
    src_string_simp = cc.convert(src_string)
with open(src_path, 'w', encoding='utf-8') as f_src:
    f_src.write(src_string_simp)

with open(src_path, 'r', encoding='utf-8') as f_src, \
     open(tgt_path, "w", encoding="utf-8") as f_tgt:
    for line in tqdm(f_src):
        f_tgt.write(encode_line(line.strip()) + '\n')

print(f"Generate complete: {os.path.getsize(src_path)/1e6:.1f}MB / {os.path.getsize(tgt_path)/1e6:.1f}MB")

处理语料:   0%|          | 0/1384748 [00:00<?, ?item/s]

处理语料: 100%|██████████| 1384748/1384748 [00:52<00:00, 26242.84item/s]
599290it [00:06, 91767.93it/s] 

Generate complete: 175.9MB / 69.1MB


## Train Sarkaz Tokenizer

In [ ]:
spm.SentencePieceTrainer.Train(
    input='corpus/train.skz', # 密文文件
    model_prefix='models/sp_skz',
    vocab_size=512,
    character_coverage=1.0,
    model_type='unigram',
    # input_sentence_size=200000,        # 采样句数
    shuffle_input_sentence=True,
    # split_by_whitespace=False,       # 密文无空格，关闭此切分
    split_by_whitespace=True,
    split_digits=False,
    num_threads=8
)
print("trained models: sp_skz.model / sp_skz.vocab")

## Train Chinese (Simplified) Tokenizer

In [ ]:
spm.SentencePieceTrainer.Train(
    input="corpus/train.zh",
    model_prefix="models/sp_zh",
    vocab_size=10000,           # 中文推荐 8000~12000
    character_coverage=0.999,   # 覆盖 99.9% 字符，生僻字自动走 <unk>
    model_type='unigram',
    # input_sentence_size=200000,
    shuffle_input_sentence=True,
    max_sentencepiece_length=16,
    num_threads=8
)
print("trained models: sp_zh.model / sp_zh.vocab")

trained models: sp_tgt.model / sp_tgt.vocab
